# Transfer learning and fine-tuning

In [ ]:
from chapter import *

Transfer learning is a technique used to leverage large models trained on a generic related task (i.e. the **pretrained model**). In this notebook, we use ResNet {cite}`resnet` which is trained to classify  [ImageNet](https://image-net.org/) consisting of 1M+ images in 1000 categories. To adapt the pretrained model to our task, we retain only the feature extractors and train a new **classification head** ({numref}`transfer-learning`).

To avoid nullifying the pretrained weights with large random gradients, we first have to train the classification head to convergence, while keeping the weights of the pretrained model fixed. Then, we proceed with **fine-tuning** where we train the entire model with a very low learning rate, again so that the pretrained weights are gradually changed.

```{figure} ../../../img/transfer-learning.png
---
name: transfer-learning
width: 80%
align: center
---
Training a new classifier over the same convolutional base. **Source:** Fig 8.12 of {cite}`keras2`.
```

PyTorch conveniently provides a collection of [vision models](https://pytorch.org/vision/main/models.html) with pretrained weights:

In [ ]:
import torchinfo
from torchvision import models

resnet = models.resnet18(pretrained=True)

BATCH_SIZE = 16
torchinfo.summary(resnet, input_size=(BATCH_SIZE, 3, 49, 49))

As described above, features from the pretrained model are passsed to a new classifier[^1]:

[^1]: Batch normalization {cite}`batchnorm` helps with activation and gradient stability. Dropout regularizes both the incoming pretrained features as well as the hidden layer.

In [ ]:
in_features = resnet.fc.in_features
num_hidden = 256

clf = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.5),
    nn.Linear(in_features, num_hidden),
    nn.ReLU(),
    nn.BatchNorm1d(num_hidden),
    nn.Dropout(0.5),
    nn.Linear(num_hidden, 2),
)

model = nn.Sequential(
    nn.Sequential(*list(resnet.children())[:-2]),
    clf
)

<br>

## Static features

Freezing the feature extraction layers:

In [ ]:
for param in model[0].parameters(): # model[0] = pretrained
    param.requires_grad = False

Setting up the data loaders[^2]:

[^2]: Recall from the previous notebook that the Dataset object applies the transform function at each indexing call (e.g. when sampling a mini-batch). Applying `Subset` gets a subset of the data based on the provided index set. Here, a contiguous index set is fine since the data has been pre-shuffled. Hence, we have the same set of images, but are augmented at each training step.

In [ ]:
train_loader = DataLoader(Subset(ds_train, torch.arange(32000)), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(Subset(ds_valid, torch.arange(8000)),  batch_size=BATCH_SIZE, shuffle=False)

Training the model using AdamW {cite}`adamw` with learning rate `0.001`:

In [ ]:
epochs = 10
optim = torch.optim.AdamW(model.parameters(), lr=0.001)
scheduler = OneCycleLR(optim, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=epochs)
trainer = Trainer(model, optim, loss_fn=F.cross_entropy, scheduler=scheduler, device=DEVICE)
trainer.run(epochs, train_loader=train_loader, valid_loader=valid_loader)

In [ ]:
%%save
def plot_training_history(trainer):
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    
    num_epochs = len(trainer.valid_log["accs"])
    num_steps_per_epoch = len(trainer.train_log["accs"]) // num_epochs
    ax[0].plot(trainer.train_log["loss"], alpha=0.3, color="C0")
    ax[1].plot(trainer.train_log["accs"], alpha=0.3, color="C0")
    ax[0].plot(trainer.train_log["loss_avg"], label="train", color="C0")
    ax[1].plot(trainer.train_log["accs_avg"], label="train", color="C0")
    ax[0].plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), trainer.valid_log["loss"], label="valid", color="C1")
    ax[1].plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), trainer.valid_log["accs"], label="valid", color="C1")
    ax[0].set_xlabel("step")
    ax[0].set_ylabel("loss")
    ax[0].grid(linestyle="dashed", alpha=0.3)
    ax[1].set_xlabel("step")
    ax[1].set_ylabel("accuracy")
    ax[1].grid(linestyle="dashed", alpha=0.3)
    ax[1].legend()
    ax[0].set_ylim(0, max(trainer.train_log["loss"]))
    ax[1].set_ylim(0, 1)
    ax[0].ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
    ax[1].ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
    
    fig.tight_layout();

In [ ]:
plot_training_history(trainer)

**Remark.** The validation step accumulates results over after an epoch for a fixed set of weights. This simulates inference performance if we load the trained model at that **checkpoint**. On the other hand, train metrics are expensive since the training dataset is large. Instead, these are accumulated at each step as an average with the previous steps.

<br>

## Fine-tuning

Unfreezing the pretrained model layers. Note that we set small learning rates:

In [ ]:
for param in model[0].parameters():
    param.requires_grad = True

# 100x smaller lr (both optim and scheduler)
epochs = 20
optim = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)
scheduler = OneCycleLR(optim, max_lr=0.0001, steps_per_epoch=len(train_loader), epochs=epochs)
trainer_ft = Trainer(model, optim, loss_fn=F.cross_entropy, scheduler=scheduler, device=DEVICE)
trainer_ft.run(epochs, train_loader=train_loader, valid_loader=valid_loader)

In [ ]:
loss = trainer.train_log["loss"] + trainer_ft.train_log["loss"]
accs = trainer.train_log["accs"] + trainer_ft.train_log["accs"]
loss_avg = trainer.train_log["loss_avg"] + trainer_ft.train_log["loss_avg"]
accs_avg = trainer.train_log["accs_avg"] + trainer_ft.train_log["accs_avg"]
val_loss = trainer.valid_log["loss"] + trainer_ft.valid_log["loss"]
val_accs = trainer.valid_log["accs"] + trainer_ft.valid_log["accs"]
num_epochs = len(val_loss)
num_steps_per_epoch = len(loss) // num_epochs

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].plot(loss, alpha=0.3, color="C0")
ax[1].plot(accs, alpha=0.3, color="C0")
ax[0].plot(loss_avg, label="train", color="C0")
ax[1].plot(accs_avg, label="train", color="C0")
ax[0].plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), val_loss, label="valid", color="C1")
ax[1].plot(list(range(num_steps_per_epoch, (num_epochs + 1) * num_steps_per_epoch, num_steps_per_epoch)), val_accs, label="valid", color="C1")
ax[0].axvline(len(trainer.train_log["loss"]), color="black", linestyle="dotted", label="[fine-tuning]")
ax[1].axvline(len(trainer.train_log["loss"]), color="black", linestyle="dotted", label="[fine-tuning]")
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].grid(linestyle="dashed", alpha=0.3)
ax[0].set_ylim(0, max(loss))
ax[1].set_xlabel("step")
ax[1].set_ylabel("accuracy")
ax[1].grid(linestyle="dashed", alpha=0.3)
ax[1].set_ylim(0, 1)
ax[1].legend(framealpha=1.0)
ax[0].ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
ax[1].ticklabel_format(axis="x", style="sci", scilimits=(3, 3))
fig.tight_layout();

Fig. *The weights of the classification head are trained on the outputs of the pretrained ResNet model with fixed weights. After the classification head forms proper weights, the pretrained weights are unfreezed, and trained with small LR. Performance improves at a faster rate, but also fluctuates more at this stage.*

<br>

**Remarks.** The model overfits and the validation curves diverge very early in the training when we turn off data augmentation (i.e. model memorizes the training data). Data augmentation prevents this by adding noise in the input. The dense layers also does not train well without BN. Finally, since the data is slightly imbalanced, we should also look at the [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix) and [PR curve](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html).

## Inference

Converting the training data loaders for batch inference:

In [ ]:
class InputDataLoader(DataLoader):
    def __init__(self, data_loader: DataLoader):
        self.data_loader = data_loader

    def __iter__(self):
        for inp, tgt in self.data_loader:
            yield inp


@torch.inference_mode()
def batch_predict(trainer: Trainer, input_loader: DataLoader):
    with eval_context(trainer.model):
        preds = [trainer(x) for x in input_loader]
        preds = torch.cat(preds, dim=0)
    return preds


pred = batch_predict(trainer, InputDataLoader(valid_loader))
print(pred.shape)
print(pred)

This should be equal to the final validation accuracy:

In [ ]:
y = torch.cat([tgt for _, tgt in valid_loader], dim=0)
print((pred.argmax(dim=1) == y.to(DEVICE)).float().mean().item())
print(trainer_ft.evaluate(valid_loader)["accs"])
print(trainer_ft.valid_log["accs"][-1]) # or look at final valid log

Note that input from our data loaders come transformed. For processing raw images, we have to transform the inputs in eval mode:

In [ ]:
file_path = "data/histopathologic-cancer-detection/test/0a0a1f3867f41e02353afcaf503f63be1bdd35ec.tif"
test_data = cv2.imread(file_path)
print(trainer.predict(transform_infer(test_data).unsqueeze(0)))

In [ ]:
PATH = "./artifacts/cancer_detection_model.pkl"
torch.save(trainer.model.state_dict(), PATH)